## Simulate Ion Channel
Copyright (c) 2025 Open Brain Institute

Author: Darshan Mandge

Last modified: 11.2025

This notebook loads an ion channel model from the Open Brain Platform and simulates it. First copy the id from `Data` Tab > `Model` > `Ion Channel Model` > Click on the one model name > on the detailed page, `Copy ID`. Replace the id in the cell below and run the notebook. 

In [ ]:
# REPLACE THE ID WITH A VALID CHANNEL ENTITY ID HERE
entity_ID = "eb971883-1ad5-4d23-a98b-9f080843977c" #staging: "6e6d1aff-c9a3-4e53-82d3-8d57424bacde" 

Load required modules and get token. 

NB: Please don't forget the to click on the output link to authenticate your token in the output of the notebook cell. The output will also ask the OBI project name to be selected. The default project name is pre-selected. Please select the corerct project if you have multiple projects.

In [ ]:
from entitysdk.models import IonChannelModel
from entitysdk.client import Client
from entitysdk.types import ContentType

from obi_auth import get_token
from obi_notebook import get_projects, get_entities
from obi_notebook.get_environment import get_environment
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)

In [ ]:
# Get the Download client from Entity SDK
client = Client(environment=get_environment(), token_manager=token)

# entities = get_entities.get_entities(token, project_context)
ion_channel = client.get_entity(
    entity_type=IonChannelModel,
    entity_id=entity_ID,
)

In [ ]:
import os
os.makedirs("./mod", exist_ok=True)
asset = client.download_assets(
    ion_channel,
    selection={"content_type": ContentType.application_mod},
    output_path="./mod",
).one()

Compile the mod files containing folder using nrnivmodl. Also, delete the old compiled mod files in the folder.

In [ ]:
! rm -r arm64
! rm -r x86_64

!nrnivmodl mod

In [ ]:
from neuron import h, gui

In [ ]:
# create a single-compartment cell to use the ion channel model
soma = h.Section("soma")
# insert ion channel
soma.insert(f"{ion_channel.nmodl_suffix}")

In [ ]:
#get the first 
try:
    for key in soma.psection()["density_mechs"][f"{ion_channel.nmodl_suffix}"]:
        # print(key)
        if "bar" in key:
            print(f"maximum conductance variable is {key}")
            # get complete variable name e.g. soma(0.5).gKv3_2bar_Kv3_2_0012
            full_attr = f"{key}_{ion_channel.nmodl_suffix}"
            print(f"Changing {full_attr} to 0.05 S/cm2")
            # set the maximum conductance variable to 0.05
            setattr(soma(0.5), full_attr, 0.05)
            print(f"Checking new value, soma(0.5).{full_attr}")
            print(getattr(soma(0.5), full_attr))
            continue

except Exception as e:
    print(e)
    print("\nPlease check the mod file and update the maximum conductance variable to an acceptable value.")
    print("Try soma(0.5).variablename_channel_SUFFIX = 0.05 or variablename_channel_SUFFIX = 0.05 to set the value")
    print("where variablename is the maximum conductance variable and channel_SUFFIX is the suffix of the ion channel from the mod file.")

In [ ]:
# create an SEClamp object (to do a voltage clamp experiment) and insert it to the centre of the soma
# https://nrn.readthedocs.io/en/latest/progref/modelspec/programmatic/mechanisms/mech.html#SEClamp
stim = h.SEClamp(soma(0.5))

# set the parameters of the SEClamp object.
# These values should be changed based on the channel used and experiment conditions
# Duration of different voltage clamp levels
stim.dur1 = 50
stim.dur2 = 100
stim.dur3 = 50

vinitial = -80
# Amplitude of different voltage clamp levels
stim.amp1 = vinitial
stim.amp2 = 0
stim.amp3 = vinitial

# set the 
stim.rs = 1e-5

In [ ]:
# create a list of voltages from -80mV to 50mV with 10mV increments
test_voltages = list(range(-80, 51, 10))

# Set up recording vectors
t = h.Vector()     # Time vector
v = h.Vector()     # Membrane potential
i = h.Vector()     # Current
t.record(h._ref_t)
v.record(soma(0.5)._ref_v)

# get the current variable name is the first variable that starts with 'i'
current_vars = [k for k in soma.psection()["density_mechs"][ion_channel.nmodl_suffix] 
               if k.startswith('i')]

# Record the channel current
i.record(getattr(soma(0.5), f'_ref_{current_vars[0]}'))

# Simulation parameters
h.dt = 0.025  # ms
h.tstop = stim.dur1 + stim.dur2 + stim.dur3

# Create figure
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes.flatten()

# Color mapping for different voltages
n_voltages = len(test_voltages)
cmap = cm.get_cmap('turbo')  # Choose: 'viridis', 'plasma', 'jet', 'coolwarm', etc.
colors = [cmap(i / (n_voltages - 1)) for i in range(n_voltages)]

# Loop over test voltages
for idx, v_test in enumerate(test_voltages):
    print(f"Running simulation for {v_test} mV")
    
    # Set test voltage
    stim.amp2 = v_test
    
    # Initialize and run simulation
    h.finitialize(vinitial)  # Initialize to vinitial
    h.run()
    
    # Convert to numpy arrays
    t_array = np.array(t)
    v_array = np.array(v)
    i_array = np.array(i) * 1e3  # Convert to pA
    
    # Plot voltage trace
    axes[0].plot(t_array, v_array, label=f'{v_test} mV', color=colors[idx])
    
    # Plot current trace
    axes[1].plot(t_array, i_array, label=f'{v_test} mV', color=colors[idx])

# Format voltage plot
axes[0].set_title('Voltage Clamp Stimulus')
axes[0].set_ylabel('Voltage (mV)')
axes[0].grid(True)
# axes[0].legend()

# Format current plot
axes[1].set_title('Voltage Clamp - Channel Current')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Current (mA/cm^2)')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()